<a href="https://colab.research.google.com/github/iknoest/fair-feeder/blob/claude%2Fcat-feeder-dataset-v13-yQuRz/smoketest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fair Feeder V13 — Smoke Test & Feeding Analysis

Run YOLOv11 inference on test images and videos from the Tapo IR camera.

**What this notebook does:**
1. Validates model accuracy on the test split (mAP, per-class AP50)
2. Benchmarks inference speed
3. Runs inference on images/videos from Google Drive
4. Tracks feeding behaviour: kibble count, cat-at-bowl, Dan's hand episodes
5. Extracts timestamps from Tapo's burned-in OSD via OCR
6. Saves text summary + snapshots to Google Drive
7. Sends results (summary, snapshots, video) to Telegram

**Secret management:** API keys are stored in [Infisical](https://app.infisical.com).
Infisical credentials (`INFISICAL_ID`, `INFISICAL_SECRET`, `INFISICAL_PROJECT_ID`) are stored in Colab Secrets.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Install dependencies
!pip install -q ultralytics roboflow easyocr tqdm requests infisical-sdk
!nvidia-smi

In [ ]:
# ── Load secrets from Infisical ───────────────────────────────────
# Infisical credentials (INFISICAL_ID, INFISICAL_SECRET, INFISICAL_PROJECT_ID)
# are stored in Colab Secrets (🔑 → Secrets in the left panel).
# The actual API keys are stored in Infisical and fetched here at runtime.
#
# Required secrets in Infisical (project → dev → /):
#   Roboflow         → Roboflow API key
#   TelegramBotToken → Telegram bot token (from @BotFather)
#   TelegramChatId   → Your Telegram chat ID

from google.colab import userdata
from infisical_sdk import InfisicalSDKClient

_client = InfisicalSDKClient(host="https://app.infisical.com")
_client.auth.universal_auth.login(
    client_id=userdata.get('INFISICAL_ID'),
    client_secret=userdata.get('INFISICAL_SECRET'),
)
_proj = userdata.get('INFISICAL_PROJECT_ID')


def _get_secret(name):
    return _client.secrets.get_secret_by_name(
        secret_name=name,
        project_id=_proj,
        environment_slug="dev",
        secret_path="/",
    ).secretValue


ROBOFLOW_API_KEY = _get_secret("Roboflow")
BOT_TOKEN        = _get_secret("TelegramBotToken")
MY_CHAT_ID       = _get_secret("TelegramChatId")

print("✅ Secrets loaded from Infisical")
print(f"   Roboflow key : {ROBOFLOW_API_KEY[:5]}...")
print(f"   Telegram bot : {BOT_TOKEN[:10]}...")

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from IPython.display import display, Video, Image as IPImage
from collections import defaultdict
from PIL import Image
import os, glob, time, re, json, requests
from tqdm.notebook import tqdm
import easyocr

In [ ]:
# Download dataset for validation metrics
# ROBOFLOW_API_KEY is loaded from Infisical in the secrets cell above

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("test-7vyqo").project("ir-kibble")
version = project.version(13)
dataset = version.download("yolov8")
DATASET_PATH = dataset.location
dataset_path = Path(DATASET_PATH)
data_yaml = str(dataset_path / "data.yaml")
print(f"\nDataset: {DATASET_PATH}")

In [ ]:
# Fix Labels: Polygon → Bounding Box
# Roboflow exports mixed polygon+bbox labels. Without this fix, YOLO drops
# segment annotations during val(), giving wrong metrics. Same fix as
# Step 3 in the training notebook.

import shutil

def is_polygon_line(line):
    return len(line.strip().split()) > 5

def polygon_to_bbox(line):
    parts = line.strip().split()
    class_id = parts[0]
    coords = [float(v) for v in parts[1:]]
    xs = coords[0::2]
    ys = coords[1::2]
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2
    width = x_max - x_min
    height = y_max - y_min
    return f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"

total_converted = 0
for split in ["train", "valid", "test"]:
    label_dir = dataset_path / split / "labels"
    if not label_dir.is_dir():
        print(f"[{split}] No labels directory — skipping")
        continue

    # Backup originals
    backup_dir = dataset_path / split / "labels_backup_polygon"
    if not backup_dir.exists():
        shutil.copytree(label_dir, backup_dir)

    # Convert
    split_converted = 0
    for f in sorted(label_dir.glob("*.txt")):
        lines = f.read_text().splitlines()
        new_lines = []
        for line in lines:
            if not line.strip():
                continue
            if is_polygon_line(line):
                new_lines.append(polygon_to_bbox(line))
                split_converted += 1
            else:
                new_lines.append(line.strip())
        f.write_text("\n".join(new_lines) + "\n" if new_lines else "")

    # Verify
    remaining = sum(
        1 for f in label_dir.glob("*.txt")
        for line in f.read_text().splitlines()
        if line.strip() and is_polygon_line(line)
    )
    status = "PASS" if remaining == 0 else f"FAIL ({remaining} polygons left)"
    print(f"[{split}] Converted {split_converted} polygon lines — Verification: {status}")
    total_converted += split_converted

print(f"\nTotal polygon lines converted: {total_converted}")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────
MODEL_PATH = '/content/drive/MyDrive/Fun Project/Cat monitor/model/fair_feeder_v13_yolov11s.pt'
SOURCE_DIR = '/content/drive/MyDrive/Fun Project/Cat monitor/Test_postmodel'
OUTPUT_DIR = '/content/drive/MyDrive/Fun Project/Cat monitor/Test_postmodel_output'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Class colours (RGB for display, BGR for OpenCV) ───────────
CLASS_COLORS_RGB = {
    "Dan":      (0,   255, 206),  # #00FFCE  teal
    "Bowl":     (134,  34, 255),  # #8622FF  purple
    "Dan_hand": (0,   16,  255),  # #0010FF  blue
    "Kibble":   (252, 244,   0),  # #FCF400  yellow
    "Sanbo":    (255, 128,   0),  # #FF8000  orange
}
CLASS_ORDER = ["Dan", "Sanbo", "Dan_hand", "Bowl", "Kibble"]

# ── Dark background colours per class (for summary table) ──────
CLASS_ROW_BG = {
    "Dan":      "#003D2F",
    "Sanbo":    "#3D2000",
    "Dan_hand": "#00073D",
    "Kibble":   "#3D3A00",
    "Bowl":     "#250050",
}

# ── Inference settings ───────────────────────────────────────────
CONF_THRESHOLD      = 0.45
IOU_THRESHOLD       = 0.20
IMGSZ               = 1280   # single int — YOLO auto-pads to stride-32
GRAY_DIFF_THRESHOLD = 15     # mean channel diff below this → IR image

# ── Behavioural thresholds ───────────────────────────────────────
OVERLAP_IOU_THRESHOLD = 0.10   # bbox overlap to count "cat at bowl" (10% IoU)

# Dan_hand model accuracy is good (AP50=0.928 in V13 validation).
# Additional filters: bowl overlap + Dan body co-detection + min duration.
DAN_HAND_CONF_THRESHOLD = 0.50   # lowered from 0.55 — bowl overlap filters false positives
DAN_HAND_GAP_SECONDS    = 2.0    # seconds without Dan_hand = new episode
DAN_HAND_MIN_SECONDS    = 0.5    # minimum episode duration to count as real

EATING_KIBBLE_DROP = 1  # min kibble count decrease to confirm eating

# ── Video processing ─────────────────────────────────────────────
FRAME_SKIP = 5  # process every 5th frame (~5 FPS effective at 27 FPS native)

# ── OCR ───────────────────────────────────────────────────────────
# Tapo C210 at 2304x1296: timestamp is larger, need bigger crop area
TIMESTAMP_CROP = (0, 0, 800, 80)  # (x1, y1, x2, y2) for Tapo timestamp region
OCR_EVERY_N_FRAMES = 30           # OCR every Nth frame (~1/sec at 25fps)

# ── File extensions ──────────────────────────────────────────────
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv"}

print("✅ Config loaded.")
print(f"   Model  : {MODEL_PATH}")
print(f"   Source : {SOURCE_DIR}")
print(f"   Output : {OUTPUT_DIR}")

In [ ]:
model = YOLO(MODEL_PATH)

# FIX: Use model.names for correct class→index mapping
# model.names = {0: 'Bowl', 1: 'Dan', 2: 'Dan_hand', 3: 'Kibble', 4: 'Sanbo'}
CLASS_NAMES = model.names
print("✅ Model loaded.")
print(f"   Classes: {CLASS_NAMES}")

In [ ]:
# FIX: imgsz=1280 (single int) avoids "must be multiple of stride 32" warnings
# FIX: Per-class AP50 uses model.names for correct index mapping
metrics = model.val(data=data_yaml, imgsz=IMGSZ, rect=True)

print(f"mAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

print("\nPer-class AP50:")
for idx in sorted(CLASS_NAMES.keys()):
    name = CLASS_NAMES[idx]
    print(f"  {name:10s} {metrics.box.ap50[idx]:.3f}")

## Model Validation — Production Acceptance Criteria

Before deploying, run the model on **held-out test videos** (not training data) and check:

### Metric thresholds

| Metric | Minimum | Good | Excellent |
|--------|---------|------|-----------|
| Overall mAP50 | ≥ 0.75 | ≥ 0.85 | ≥ 0.90 |
| Per-class AP50 (Dan, Sanbo, Bowl) | ≥ 0.80 | ≥ 0.90 | ≥ 0.95 |
| Per-class AP50 (Kibble) | ≥ 0.65 | ≥ 0.80 | ≥ 0.90 |
| Per-class AP50 (Dan_hand) | ≥ 0.70 | ≥ 0.85 | ≥ 0.92 |
| False positive rate | < 15% | < 8% | < 3% |

### Scenario testing checklist

For each scenario below, run `model.predict()` on a representative video clip and manually verify detection correctness (count true positives, false positives, and missed detections):

1. **Normal feeding** — both cats at bowl, kibble visible
2. **Sanbo only** — no Dan in frame
3. **Dan_hand feeding** — hand placing kibble into bowl
4. **Empty bowl** — no cats, no kibble (check for false positives)
5. **Night IR mode** — very dark, IR-only footage
6. **Motion blur** — cat moving quickly past bowl
7. **Two cats overlapping** — both cats touching at bowl simultaneously
8. **Dan_hand partially out of frame** — hand at edge of camera view

### What to watch in the confusion matrix

- **Kibble ↔ Bowl** confusion → inflates kibble count → wrong eating estimates
- **Dan_hand → Dan** misclassification → missed hand-feeding events
- **Dan ↔ Sanbo** confusion → wrong attribution of who ate how much

### Suggested workflow

1. Collect 5–10 short clips (30–60 sec each) covering the scenarios above
2. Run the full pipeline on each clip (this notebook)
3. Compare printed summary vs what you see in the video
4. Track pass/fail per scenario — aim for **≥ 7/8 scenarios passing**
5. If a scenario fails, add more training images for that case and retrain

In [ ]:
# Benchmark inference speed
# FIX: imgsz=1280 (single int)
test_img = str(next(Path(DATASET_PATH, "test", "images").glob("*.jpg")))

# Warmup
for _ in range(5):
    model.predict(test_img, imgsz=IMGSZ, verbose=False)

# Benchmark
times = []
for _ in range(50):
    t0 = time.perf_counter()
    model.predict(test_img, imgsz=IMGSZ, verbose=False)
    times.append(time.perf_counter() - t0)

avg_ms = sum(times) / len(times) * 1000
fps = 1000 / avg_ms
print(f"Avg inference: {avg_ms:.1f} ms  ({fps:.1f} FPS)")

In [ ]:
# ───────────────────────────────────────────────────────────────
# OCR + Helper Functions
# ───────────────────────────────────────────────────────────────

# --- OCR reader for Tapo timestamp ---
ocr_reader = easyocr.Reader(['en'], gpu=False)
TIMESTAMP_REGEX = re.compile(r'(\d{4}[-/]\d{2}[-/]\d{2})\s*(\d{1,2}:\d{2}:\d{1,2})')


def extract_timestamp(frame):
    """Crop top-left corner, OCR the Tapo burned-in timestamp."""
    x1, y1, x2, y2 = TIMESTAMP_CROP
    crop = frame[y1:y2, x1:x2]
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY)
    results = ocr_reader.readtext(thresh, detail=0)
    # Strip all spaces — easyOCR often reads chars individually: "2 0 2 6 - 0 1"
    text = "".join(results).replace(" ", "")
    m = TIMESTAMP_REGEX.search(text)
    if m:
        date_part = m.group(1).replace("/", "-")
        time_part = m.group(2)
        # Zero-pad partial components: H:MM:S → HH:MM:SS
        h, mi, s = time_part.split(":")
        time_part = f"{h.zfill(2)}:{mi}:{s.zfill(2)}"
        return f"{date_part} {time_part}"
    return text


def classify_file(filepath):
    """Return 'image', 'video', or None based on file extension."""
    ext = Path(filepath).suffix.lower()
    if ext in IMAGE_EXTENSIONS:
        return "image"
    elif ext in VIDEO_EXTENSIONS:
        return "video"
    return None


def detect_image_type(img_bgr):
    """
    Returns 'color' or 'ir'.
    Tapo IR images are grayscale stored as BGR where R≈G≈B.
    """
    b, g, r = cv2.split(img_bgr)
    b, g, r = b.astype(int), g.astype(int), r.astype(int)
    max_diff = max(
        np.mean(np.abs(r - g)),
        np.mean(np.abs(r - b)),
        np.mean(np.abs(g - b)),
    )
    return "ir" if max_diff < GRAY_DIFF_THRESHOLD else "color"


def prepare_for_inference(img_bgr, image_type):
    """Color image → gray then back to 3-ch BGR for YOLO. IR → pass through."""
    if image_type == "color":
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    return img_bgr


def get_color_bgr(class_name):
    r, g, b = CLASS_COLORS_RGB.get(class_name, (255, 255, 255))
    return (b, g, r)


def bbox_iou(box_a, box_b):
    """Compute IoU between two dicts with x1,y1,x2,y2 keys."""
    xa1 = max(box_a["x1"], box_b["x1"])
    ya1 = max(box_a["y1"], box_b["y1"])
    xa2 = min(box_a["x2"], box_b["x2"])
    ya2 = min(box_a["y2"], box_b["y2"])
    inter = max(0, xa2 - xa1) * max(0, ya2 - ya1)
    area_a = (box_a["x2"] - box_a["x1"]) * (box_a["y2"] - box_a["y1"])
    area_b = (box_b["x2"] - box_b["x1"]) * (box_b["y2"] - box_b["y1"])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def parse_results(results):
    """Parse YOLO results into a list of detection dicts."""
    detections = []
    for result in results:
        if result.boxes is None:
            continue
        for box in result.boxes:
            cls_id = int(box.cls.item())
            detections.append({
                "class_name": model.names[cls_id],
                "conf": float(box.conf.item()),
                "x1": int(box.xyxy[0][0].item()),
                "y1": int(box.xyxy[0][1].item()),
                "x2": int(box.xyxy[0][2].item()),
                "y2": int(box.xyxy[0][3].item()),
            })
    return detections


def draw_boxes(img_bgr, detections, show_label=False):
    """Draw boxes on a copy of img_bgr."""
    out = img_bgr.copy()
    thick = max(2, int(img_bgr.shape[1] / 600))
    for det in detections:
        name = det["class_name"]
        conf = det["conf"]
        x1, y1, x2, y2 = det["x1"], det["y1"], det["x2"], det["y2"]
        bgr = get_color_bgr(name)
        cv2.rectangle(out, (x1, y1), (x2, y2), bgr, thick)
        if show_label:
            label = f"{name} {conf*100:.0f}%"
            font = cv2.FONT_HERSHEY_SIMPLEX
            fscale = max(0.5, img_bgr.shape[1] / 2000)
            fthick = max(1, thick - 1)
            (tw, th), _ = cv2.getTextSize(label, font, fscale, fthick)
            pad = 4
            cv2.rectangle(out, (x1, max(0, y1 - th - pad*2)), (x1 + tw + pad, y1), bgr, -1)
            cv2.putText(out, label, (x1 + pad//2, y1 - pad), font, fscale, (0,0,0), fthick, cv2.LINE_AA)
    return out


def compute_stats(values):
    if not values:
        return {"count": 0, "min": None, "max": None, "median": None, "variance": None}
    arr = np.array(values, dtype=float)
    return {
        "count": len(arr),
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
        "median": float(np.median(arr)),
        "variance": float(np.var(arr)),
    }


def print_image_summary(img_name, image_type, stats_by_class):
    tag = "🌈 COLOR (converted to gray for inference)" if image_type == "color" else "🌙 IR"
    print(f"\n{'─'*65}")
    print(f"  📷  {img_name}  [{tag}]")
    print(f"{'─'*65}")
    print(f"  {'Class':<12} {'Count':>6}  {'Min%':>6}  {'Max%':>6}  {'Med%':>6}  {'Variance':>9}")
    print(f"  {'─'*59}")
    total = 0
    for name in CLASS_ORDER:
        s = stats_by_class.get(name, compute_stats([]))
        if s["count"] > 0:
            print(f"  {name:<12} {s['count']:>6}  "
                  f"{s['min']*100:>5.1f}%  "
                  f"{s['max']*100:>5.1f}%  "
                  f"{s['median']*100:>5.1f}%  "
                  f"{s['variance']*10000:>8.2f}")
            total += s["count"]
        else:
            print(f"  {name:<12} {'—':>6}")
    print(f"  {'─'*59}")
    print(f"  {'TOTAL':<12} {total:>6}")


def show_dual_output(img_name, image_type, img_plain, img_labeled, stats_by_class):
    left = cv2.cvtColor(img_plain, cv2.COLOR_BGR2RGB)
    right = cv2.cvtColor(img_labeled, cv2.COLOR_BGR2RGB)
    tag = "COLOR → gray inference" if image_type == "color" else "IR"
    fig, axes = plt.subplots(1, 2, figsize=(22, 7), facecolor="#1a1a1a")
    for ax, img, title in zip(axes, [left, right], ["Boxes only", "Boxes + Confidence"]):
        ax.imshow(img)
        ax.set_title(title, color="white", fontsize=12, pad=8)
        ax.axis("off")
    fig.suptitle(f"{img_name}  [{tag}]", color="white", fontsize=13, y=1.01, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"preview_{Path(img_name).stem}.jpg"),
                dpi=120, bbox_inches="tight", facecolor="#1a1a1a")
    plt.show()
    plt.close()
    print_image_summary(img_name, image_type, stats_by_class)


def render_final_table(total_images, color_count, ir_count, all_stats):
    """Styled summary table saved as SUMMARY_TABLE.jpg."""
    col_labels = ["Class", "Detections", "Min %", "Max %", "Median %", "Variance"]
    col_widths = [0.18, 0.14, 0.12, 0.12, 0.14, 0.14]
    col_align = ["left", "center", "center", "center", "center", "center"]
    rows = []
    row_meta = []
    grand_all = []
    for name in CLASS_ORDER:
        confs = all_stats.get(name, [])
        grand_all.extend(confs)
        s = compute_stats(confs)
        rows.append([
            name,
            str(s["count"]),
            f"{s['min']*100:.1f}%" if s["count"] else "—",
            f"{s['max']*100:.1f}%" if s["count"] else "—",
            f"{s['median']*100:.1f}%" if s["count"] else "—",
            f"{s['variance']*10000:.2f}" if s["count"] else "—",
        ])
        row_meta.append((name, CLASS_ROW_BG.get(name, "#1E1E1E")))
    gs = compute_stats(grand_all)
    rows.append([
        "ALL CLASSES",
        str(gs["count"]),
        f"{gs['min']*100:.1f}%" if gs["count"] else "—",
        f"{gs['max']*100:.1f}%" if gs["count"] else "—",
        f"{gs['median']*100:.1f}%" if gs["count"] else "—",
        f"{gs['variance']*10000:.2f}" if gs["count"] else "—",
    ])
    row_meta.append(("__total__", "#2C2C2C"))
    n_data_rows = len(rows)
    row_h = 0.11
    header_h = 0.13
    top_pad = 0.25
    bottom_pad = 0.12
    total_h = top_pad + header_h + n_data_rows * row_h + bottom_pad
    fig, ax = plt.subplots(figsize=(13, total_h * 6), facecolor="#1a1a1a")
    ax.set_facecolor("#1a1a1a")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, total_h)
    ax.axis("off")
    ax.text(0.5, total_h - 0.05, "Fair Feeder V13 — Smoke Test Summary",
            ha="center", va="top", color="white", fontsize=14, fontweight="bold")
    ax.text(0.5, total_h - 0.13,
            f"Images analysed: {total_images}     🌈 Color: {color_count}     🌙 IR: {ir_count}",
            ha="center", va="top", color="#AAAAAA", fontsize=10)
    x_starts = []
    x = 0.01
    for w in col_widths:
        x_starts.append(x)
        x += w
    header_y = total_h - top_pad
    header_rect = patches.FancyBboxPatch(
        (0.01, header_y - header_h + 0.02), 0.97, header_h,
        boxstyle="round,pad=0.005", linewidth=0, facecolor="#1F4E78")
    ax.add_patch(header_rect)
    for xi, w, label, align in zip(x_starts, col_widths, col_labels, col_align):
        tx = xi + w/2 if align == "center" else xi + 0.01
        ax.text(tx, header_y - header_h/2 + 0.02, label,
                ha=align if align != "left" else "left", va="center",
                color="white", fontsize=10, fontweight="bold")
    for i, (row, (rname, bg)) in enumerate(zip(rows, row_meta)):
        y_top = header_y - header_h - i * row_h
        is_total = (rname == "__total__")
        rect = patches.FancyBboxPatch(
            (0.01, y_top - row_h + 0.005), 0.97, row_h - 0.01,
            boxstyle="round,pad=0.003", linewidth=0, facecolor=bg)
        ax.add_patch(rect)
        for j, (xi, w, val, align) in enumerate(zip(x_starts, col_widths, row, col_align)):
            tx = xi + w/2 if align == "center" else xi + 0.01
            if j == 0:
                r_c, g_c, b_c = CLASS_COLORS_RGB.get(rname, (255, 215, 0)) if not is_total else (255, 215, 0)
                txt_color = "#%02X%02X%02X" % (r_c, g_c, b_c) if not is_total else "#FFD700"
            else:
                txt_color = "#FFD700" if is_total else "white"
            ax.text(tx, y_top - row_h/2 + 0.005, val,
                    ha="left" if align == "left" else "center", va="center",
                    color=txt_color, fontsize=10,
                    fontweight="bold" if is_total else "normal")
    ax.text(0.5, 0.04,
            f"conf_threshold={CONF_THRESHOLD}  |  iou_threshold={IOU_THRESHOLD}  |  imgsz={IMGSZ}  |  Variance × 10⁴",
            ha="center", va="bottom", color="#555555", fontsize=8)
    table_path = os.path.join(OUTPUT_DIR, "SUMMARY_TABLE.jpg")
    plt.savefig(table_path, dpi=150, bbox_inches="tight", facecolor="#1a1a1a")
    plt.show()
    plt.close()
    print(f"\n✅  Summary table saved → {table_path}")


print("✅ Helper functions defined.")

In [ ]:
# ───────────────────────────────────────────────────────────────
# Feeding Behaviour Tracker
# ───────────────────────────────────────────────────────────────

class FeedingTracker:
    """Tracks feeding events across video frames.

    Eating logic — phase-based attribution:
      1. Merge overlapping cat-at-bowl episodes into "feeding phases"
      2. Measure kibble only at clear frames (no cat/hand occluding bowl)
      3. Attribute kibble drops to cat(s) present, proportional to
         their bowl-overlap time within each phase
    """

    def __init__(self, fps=25.0):
        self.fps = fps
        self.dan_hand_gap_frames = int(DAN_HAND_GAP_SECONDS * fps)
        self.dan_hand_min_frames = int(DAN_HAND_MIN_SECONDS * fps)

        # Per-frame arrays
        self.kibble_counts = []       # int per frame
        self.dan_at_bowl = []         # bool per frame
        self.sanbo_at_bowl = []       # bool per frame
        self.dan_hand_present = []    # bool per frame (filtered: bowl overlap + conf + Dan body)
        self.dan_hand_confs = []      # float per frame (max conf, 0 if no valid hand)
        self.timestamps = []          # OCR string per frame

        # Events
        self.dan_first_frame = None
        self.dan_first_timestamp = None
        self.sanbo_first_frame = None
        self.sanbo_first_timestamp = None
        self.snapshots = {}  # {"sanbo_arrival": img, "dan_hand_ep0": img, ...}

        # Internal: track best-confidence frame per Dan_hand episode
        self._current_hand_ep_idx = -1
        self._current_hand_best_conf = 0.0

        # Kibble-dispensed snapshot: watch for stable kibble after hand episode
        self._kibble_watch_active = False
        self._kibble_watch_ep_idx = 0
        self._kibble_watch_stable_buf = []  # list of (frame_idx, kibble_count, frame_copy)

    def process_frame(self, frame_idx, detections, timestamp_str, frame_img):
        """Process one frame's detections."""
        # Kibble count
        kibble_count = sum(1 for d in detections if d["class_name"] == "Kibble")
        self.kibble_counts.append(kibble_count)
        self.timestamps.append(timestamp_str)

        # Find Bowl bbox (use largest if multiple)
        bowl_boxes = [d for d in detections if d["class_name"] == "Bowl"]
        bowl_box = max(bowl_boxes, key=lambda d: (d["x2"]-d["x1"])*(d["y2"]-d["y1"])) if bowl_boxes else None

        # Cat-at-bowl overlap (IoU > 10%)
        dan_at = False
        sanbo_at = False
        if bowl_box:
            for d in detections:
                if d["class_name"] == "Dan" and bbox_iou(d, bowl_box) > OVERLAP_IOU_THRESHOLD:
                    dan_at = True
                elif d["class_name"] == "Sanbo" and bbox_iou(d, bowl_box) > OVERLAP_IOU_THRESHOLD:
                    sanbo_at = True
        self.dan_at_bowl.append(dan_at)
        self.sanbo_at_bowl.append(sanbo_at)

        # Dan body presence check (used for Dan_hand co-detection + first appearance)
        dan_here = any(d["class_name"] == "Dan" for d in detections)

        # Dan_hand: require conf + bowl overlap + Dan body in same frame
        dan_hand = False
        dan_hand_conf = 0.0
        if bowl_box and dan_here:
            for d in detections:
                if d["class_name"] == "Dan_hand" and d["conf"] >= DAN_HAND_CONF_THRESHOLD:
                    if bbox_iou(d, bowl_box) > OVERLAP_IOU_THRESHOLD:
                        dan_hand = True
                        dan_hand_conf = max(dan_hand_conf, d["conf"])
        self.dan_hand_present.append(dan_hand)
        self.dan_hand_confs.append(dan_hand_conf)

        # Dan first appearance (informational — anywhere in frame)
        if dan_here and self.dan_first_frame is None:
            self.dan_first_frame = frame_idx
            self.dan_first_timestamp = timestamp_str

        # Sanbo arrival = first frame where Sanbo overlaps Bowl (not just detected)
        if sanbo_at and self.sanbo_first_frame is None:
            self.sanbo_first_frame = frame_idx
            self.sanbo_first_timestamp = timestamp_str
            annotated_snap = draw_boxes(frame_img, detections, show_label=True)
            self.snapshots["sanbo_arrival"] = annotated_snap

        # Dan_hand snapshot: capture highest-confidence frame per episode
        if dan_hand:
            prev_hand = self.dan_hand_present[-2] if len(self.dan_hand_present) >= 2 else False
            if not prev_hand:
                # New episode starting
                ep_idx = self._count_hand_episodes_so_far()
                self._current_hand_ep_idx = ep_idx
                self._current_hand_best_conf = dan_hand_conf
                annotated_snap = draw_boxes(frame_img, detections, show_label=True)
                self.snapshots[f"dan_hand_ep{ep_idx}"] = annotated_snap
            else:
                # Continuing episode: update snapshot if higher confidence
                if dan_hand_conf > self._current_hand_best_conf:
                    self._current_hand_best_conf = dan_hand_conf
                    annotated_snap = draw_boxes(frame_img, detections, show_label=True)
                    self.snapshots[f"dan_hand_ep{self._current_hand_ep_idx}"] = annotated_snap

        # ── Kibble-dispensed snapshot logic ────────────────────
        # Detect falling edge of Dan_hand (episode just ended)
        if len(self.dan_hand_present) >= 2 and \
           not self.dan_hand_present[-1] and self.dan_hand_present[-2]:
            self._kibble_watch_active = True
            self._kibble_watch_ep_idx = self._current_hand_ep_idx
            self._kibble_watch_stable_buf = []

        # Accumulate clear frames post-dispense to find stable kibble peak
        if self._kibble_watch_active:
            if not dan_at and not sanbo_at and not dan_hand:
                self._kibble_watch_stable_buf.append(
                    (frame_idx, kibble_count, frame_img.copy(), detections))
                if len(self._kibble_watch_stable_buf) >= 3:
                    recent = self._kibble_watch_stable_buf[-3:]
                    counts = [c for _, c, _, _ in recent]
                    # Stable = all 3 recent clear counts are identical and > 0
                    if len(set(counts)) == 1 and counts[0] > 0:
                        ep_key = f"kibble_dispensed_ep{self._kibble_watch_ep_idx}"
                        if ep_key not in self.snapshots:
                            snap_frame = recent[0][2]  # first of the 3 stable frames
                            snap_dets = recent[0][3]
                            self.snapshots[ep_key] = draw_boxes(
                                snap_frame, snap_dets, show_label=False)
                        self._kibble_watch_active = False
            else:
                # Cat/hand arrived before stabilising — reset buffer, keep watching
                self._kibble_watch_stable_buf = []

    def _count_hand_episodes_so_far(self):
        """Count how many Dan_hand episodes have started so far."""
        episodes = 0
        in_episode = False
        gap = 0
        for present in self.dan_hand_present:
            if present:
                if not in_episode:
                    episodes += 1
                    in_episode = True
                gap = 0
            else:
                gap += 1
                if gap >= self.dan_hand_gap_frames:
                    in_episode = False
        return episodes - 1  # current one is starting, so index = count-1

    def _find_episodes(self, bool_array, gap_frames):
        """Find contiguous True episodes with a minimum gap between them.
        Returns list of (start_frame, end_frame)."""
        episodes = []
        start = None
        gap = 0
        for i, val in enumerate(bool_array):
            if val:
                if start is None:
                    start = i
                gap = 0
            else:
                gap += 1
                if start is not None and gap >= gap_frames:
                    episodes.append((start, i - gap))
                    start = None
        if start is not None:
            episodes.append((start, len(bool_array) - 1))
        return episodes

    def _find_clear_kibble_count(self, from_frame, direction="before"):
        """Search for a reliable kibble count from clear frames.

        A 'clear frame' = no cat and no hand bbox overlaps the bowl.
        Searches up to 5 seconds in the given direction. Collects up to
        5 consecutive clear frames for a stable median.

        Returns int or None (if no clear frames found).
        """
        max_search = int(self.fps * 5)
        n = len(self.kibble_counts)

        if direction == "before":
            search_range = range(from_frame - 1, max(from_frame - max_search, -1), -1)
        else:
            search_range = range(from_frame + 1, min(from_frame + max_search, n))

        clear_counts = []
        for i in search_range:
            if 0 <= i < n:
                if not self.dan_at_bowl[i] and not self.sanbo_at_bowl[i] and not self.dan_hand_present[i]:
                    clear_counts.append(self.kibble_counts[i])
                    if len(clear_counts) >= 5:
                        break
                elif clear_counts:
                    # Had clear frames, now hit occluded — stop
                    break

        return int(np.median(clear_counts)) if clear_counts else None

    def _build_feeding_phases(self):
        """Merge all cat-at-bowl episodes into contiguous feeding phases.

        A feeding phase = contiguous period where at least one cat is at
        the bowl. Small gaps (< 1 second) are bridged.

        Returns list of dicts with start, end, dan_frames, sanbo_frames.
        """
        n = len(self.kibble_counts)
        if n == 0:
            return []

        any_at_bowl = [
            self.dan_at_bowl[i] or self.sanbo_at_bowl[i]
            for i in range(n)
        ]

        gap_tolerance = int(self.fps * 1.0)
        raw_phases = self._find_episodes(any_at_bowl, gap_frames=gap_tolerance)

        phases = []
        for start_f, end_f in raw_phases:
            dan_count = sum(1 for i in range(start_f, end_f + 1) if self.dan_at_bowl[i])
            sanbo_count = sum(1 for i in range(start_f, end_f + 1) if self.sanbo_at_bowl[i])
            phases.append({
                'start': start_f,
                'end': end_f,
                'dan_frames': dan_count,
                'sanbo_frames': sanbo_count,
            })

        return phases

    def _get_timestamp(self, frame_idx):
        """Get the closest non-empty timestamp for a frame."""
        if frame_idx < len(self.timestamps) and self.timestamps[frame_idx]:
            return self.timestamps[frame_idx]
        for offset in range(1, 60):
            for idx in [frame_idx - offset, frame_idx + offset]:
                if 0 <= idx < len(self.timestamps) and self.timestamps[idx]:
                    return self.timestamps[idx]
        return "??"

    def summarize(self):
        """Compute feeding summary using phase-based eating attribution."""
        n_frames = len(self.kibble_counts)
        if n_frames == 0:
            return {"error": "No frames processed"}

        # ── Smooth kibble counts (rolling median, window=3) ───
        # Reduces flicker from same kibbles being detected/undetected
        if n_frames >= 3:
            smoothed = []
            for i in range(n_frames):
                w_start = max(0, i - 1)
                w_end = min(n_frames, i + 2)
                smoothed.append(int(np.median(self.kibble_counts[w_start:w_end])))
            self.kibble_counts = smoothed

        start_ts = self._get_timestamp(0)
        end_ts = self._get_timestamp(n_frames - 1)

        # ── Dan_hand episodes ─────────────────────────────────
        hand_episodes = self._find_episodes(self.dan_hand_present, self.dan_hand_gap_frames)
        # Filter out episodes shorter than minimum duration
        hand_episodes = [
            (s, e) for s, e in hand_episodes
            if (e - s) >= self.dan_hand_min_frames
        ]

        # Clean up orphaned snapshots for filtered-out episodes
        confirmed_ep_indices = set(range(len(hand_episodes)))
        orphan_keys = [k for k in list(self.snapshots.keys())
                       if k.startswith("dan_hand_ep")
                       and int(k.split("ep")[1]) not in confirmed_ep_indices]
        for k in orphan_keys:
            del self.snapshots[k]
        # Also clean up kibble_dispensed snapshots for orphaned episodes
        orphan_kibble_keys = [k for k in list(self.snapshots.keys())
                              if k.startswith("kibble_dispensed_ep")
                              and int(k.split("ep")[1]) not in confirmed_ep_indices]
        for k in orphan_kibble_keys:
            del self.snapshots[k]

        hand_summary = []
        for start_f, end_f in hand_episodes:
            kb_before = self._find_clear_kibble_count(start_f, direction="before")
            kb_after = self._find_clear_kibble_count(end_f, direction="after")
            kibble_added = 0
            if kb_before is not None and kb_after is not None:
                kibble_added = max(0, kb_after - kb_before)
            hand_summary.append({
                "start_frame": start_f,
                "end_frame": end_f,
                "timestamp": self._get_timestamp(start_f),
                "kibble_added": kibble_added,
            })

        # ── Phase-based eating attribution ────────────────────
        phases = self._build_feeding_phases()

        dan_kibble_eaten = 0
        sanbo_kibble_eaten = 0
        dan_bowl_seconds = 0.0
        sanbo_bowl_seconds = 0.0

        for phase in phases:
            start_f = phase['start']
            end_f = phase['end']
            dan_f = phase['dan_frames']
            sanbo_f = phase['sanbo_frames']

            dan_bowl_seconds += dan_f / self.fps
            sanbo_bowl_seconds += sanbo_f / self.fps

            kb_before = self._find_clear_kibble_count(start_f, direction="before")
            kb_after = self._find_clear_kibble_count(end_f, direction="after")

            # Edge case: video starts during feeding, no clear frames before
            if kb_before is None and start_f < int(self.fps * 2):
                kb_before = self.kibble_counts[0]
            # Edge case: video ends during feeding, no clear frames after
            if kb_after is None and end_f > n_frames - int(self.fps * 2):
                kb_after = self.kibble_counts[-1]

            if kb_before is None or kb_after is None:
                continue

            drop = kb_before - kb_after
            if drop < EATING_KIBBLE_DROP:
                continue

            # Attribute to cat(s) present at the bowl during this phase
            if dan_f > 0 and sanbo_f == 0:
                dan_kibble_eaten += drop
            elif sanbo_f > 0 and dan_f == 0:
                sanbo_kibble_eaten += drop
            elif dan_f > 0 and sanbo_f > 0:
                total_f = dan_f + sanbo_f
                dan_kibble_eaten += round(drop * dan_f / total_f)
                sanbo_kibble_eaten += round(drop * sanbo_f / total_f)

        # ── Double-counting guard ─────────────────────────────
        first_clear = self._find_clear_kibble_count(0, direction="after")
        last_clear = self._find_clear_kibble_count(n_frames - 1, direction="before")
        if first_clear is not None and last_clear is not None:
            max_total = max(0, first_clear - last_clear)
            total_attr = dan_kibble_eaten + sanbo_kibble_eaten
            if total_attr > max_total and total_attr > 0:
                scale = max_total / total_attr
                dan_kibble_eaten = round(dan_kibble_eaten * scale)
                sanbo_kibble_eaten = round(sanbo_kibble_eaten * scale)

        return {
            "n_frames": n_frames,
            "duration_sec": n_frames / self.fps,
            "start_ts": start_ts,
            "end_ts": end_ts,
            "dan_first_ts": self.dan_first_timestamp,
            "sanbo_first_ts": self.sanbo_first_timestamp,
            "hand_episodes": hand_summary,
            "dan_kibble_eaten": dan_kibble_eaten,
            "dan_bowl_seconds": dan_bowl_seconds,
            "sanbo_kibble_eaten": sanbo_kibble_eaten,
            "sanbo_bowl_seconds": sanbo_bowl_seconds,
        }


def format_duration(seconds):
    m, s = divmod(int(seconds), 60)
    return f"{m}m {s:02d}s"


def format_feeding_summary(summary, video_name):
    """Format a summary dict into a readable string (for print + Telegram)."""
    lines = []
    lines.append(f"\U0001f431 Fair Feeder Summary \u2014 {video_name}")
    lines.append(f"   {summary['start_ts']} \u2192 {summary['end_ts']}")
    lines.append(f"   Duration: {format_duration(summary['duration_sec'])}")
    lines.append("\u2501" * 48)

    # Dan_hand feeding
    hand_eps = summary["hand_episodes"]
    lines.append(f"\n\U0001f4e6 Feeding (Dan_hand): {len(hand_eps)} attempt(s)")
    for ep in hand_eps:
        lines.append(f"   {ep['timestamp']} \u2014 ~{ep['kibble_added']} kibble added")

    # Cat arrivals
    if summary["sanbo_first_ts"]:
        lines.append(f"\n\U0001f408 Sanbo arrived at bowl: {summary['sanbo_first_ts']}")
    else:
        lines.append("\n\U0001f408 Sanbo: not detected at bowl")
    if summary["dan_first_ts"]:
        lines.append(f"\U0001f408\u200d\u2b1b Dan present from {summary['dan_first_ts']}")
    else:
        lines.append("\U0001f408\u200d\u2b1b Dan: not detected")

    # Eating estimates
    lines.append(f"\n\U0001f37d\ufe0f Eating estimate:")
    lines.append(f"   Dan:   ~{summary['dan_kibble_eaten']} kibble (at bowl {format_duration(summary['dan_bowl_seconds'])})")
    lines.append(f"   Sanbo: ~{summary['sanbo_kibble_eaten']} kibble (at bowl {format_duration(summary['sanbo_bowl_seconds'])})")

    # Dan_hand attempts
    lines.append(f"\n\u270b Dan's hand attempts: {len(hand_eps)}")

    return "\n".join(lines)


def plot_video_timeline(tracker, video_name):
    """Plot timeline chart: kibble count, cat presence, Dan_hand events."""
    n = len(tracker.kibble_counts)
    if n == 0:
        return
    frames = np.arange(n)
    time_sec = frames / tracker.fps

    fig, axes = plt.subplots(4, 1, figsize=(16, 8), sharex=True, facecolor="#1a1a1a")
    fig.suptitle(f"Detection Timeline: {video_name}", color="white", fontsize=13, fontweight="bold")

    # Kibble count
    axes[0].fill_between(time_sec, tracker.kibble_counts, color="#FCF400", alpha=0.7)
    axes[0].set_ylabel("Kibble", color="white", fontsize=10)
    axes[0].set_facecolor("#1a1a1a")
    axes[0].tick_params(colors="white")

    # Dan at bowl
    dan_arr = np.array(tracker.dan_at_bowl, dtype=float)
    axes[1].fill_between(time_sec, dan_arr, color="#00FFCE", alpha=0.7, step="mid")
    axes[1].set_ylabel("Dan\nat bowl", color="white", fontsize=10)
    axes[1].set_ylim(-0.1, 1.1)
    axes[1].set_facecolor("#1a1a1a")
    axes[1].tick_params(colors="white")

    # Sanbo at bowl
    sanbo_arr = np.array(tracker.sanbo_at_bowl, dtype=float)
    axes[2].fill_between(time_sec, sanbo_arr, color="#FF8000", alpha=0.7, step="mid")
    axes[2].set_ylabel("Sanbo\nat bowl", color="white", fontsize=10)
    axes[2].set_ylim(-0.1, 1.1)
    axes[2].set_facecolor("#1a1a1a")
    axes[2].tick_params(colors="white")

    # Dan_hand
    hand_arr = np.array(tracker.dan_hand_present, dtype=float)
    axes[3].fill_between(time_sec, hand_arr, color="#0010FF", alpha=0.7, step="mid")
    axes[3].set_ylabel("Dan\nhand", color="white", fontsize=10)
    axes[3].set_ylim(-0.1, 1.1)
    axes[3].set_xlabel("Time (seconds)", color="white", fontsize=10)
    axes[3].set_facecolor("#1a1a1a")
    axes[3].tick_params(colors="white")

    plt.tight_layout()
    timeline_path = os.path.join(OUTPUT_DIR, f"timeline_{Path(video_name).stem}.jpg")
    plt.savefig(timeline_path, dpi=120, bbox_inches="tight", facecolor="#1a1a1a")
    plt.show()
    plt.close()
    print(f"\u2705 Timeline saved \u2192 {timeline_path}")


print("✅ FeedingTracker defined.")

In [ ]:
# ───────────────────────────────────────────────────────────────
# Scan SOURCE_DIR + Process Images
# ───────────────────────────────────────────────────────────────

all_files = sorted(Path(SOURCE_DIR).iterdir())
image_paths = [f for f in all_files if classify_file(f) == "image"]
video_paths = [f for f in all_files if classify_file(f) == "video"]

print(f"✅ Found {len(image_paths)} image(s) and {len(video_paths)} video(s) in SOURCE_DIR")

if not image_paths:
    print(f"   No images found in: {SOURCE_DIR}")
else:
    print(f"\n{'='*65}")
    print(f"  Processing {len(image_paths)} image(s)...")
    print(f"{'='*65}")

    all_stats = {}
    color_count = 0
    ir_count = 0

    for img_path in image_paths:
        img_name = img_path.name
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            print(f"\u26a0\ufe0f  Could not read: {img_name} \u2014 skipping")
            continue

        image_type = detect_image_type(img_bgr)
        if image_type == "color":
            color_count += 1
        else:
            ir_count += 1

        inference_img = prepare_for_inference(img_bgr, image_type)
        tmp_path = "/tmp/_infer_tmp.jpg"
        cv2.imwrite(tmp_path, inference_img)

        results = model.predict(
            source=tmp_path,
            imgsz=IMGSZ,
            conf=CONF_THRESHOLD,
            iou=IOU_THRESHOLD,
            rect=True,
            verbose=False,
        )

        detections = parse_results(results)
        confs_by_class = {}
        for det in detections:
            confs_by_class.setdefault(det["class_name"], []).append(det["conf"])

        stats_by_class = {name: compute_stats(confs) for name, confs in confs_by_class.items()}
        for name, confs in confs_by_class.items():
            all_stats.setdefault(name, []).extend(confs)

        img_plain = draw_boxes(img_bgr, detections, show_label=False)
        img_labeled = draw_boxes(img_bgr, detections, show_label=True)

        stem = Path(img_name).stem
        cv2.imwrite(os.path.join(OUTPUT_DIR, f"{stem}_boxes_only.jpg"), img_plain)
        cv2.imwrite(os.path.join(OUTPUT_DIR, f"{stem}_with_conf.jpg"), img_labeled)

        show_dual_output(img_name, image_type, img_plain, img_labeled, stats_by_class)

    # Final summary table for all images
    if all_stats:
        render_final_table(
            total_images=len(image_paths),
            color_count=color_count,
            ir_count=ir_count,
            all_stats=all_stats,
        )

    print(f"\n✅  Image outputs saved to: {OUTPUT_DIR}")

In [ ]:
# ───────────────────────────────────────────────────────────────
# Process Videos
# ───────────────────────────────────────────────────────────────

# Re-scan SOURCE_DIR so this cell works even if run independently
all_files = sorted(Path(SOURCE_DIR).iterdir())
video_paths = [f for f in all_files if classify_file(f) == "video"]
print(f"✅ Found {len(video_paths)} video(s) in SOURCE_DIR")

video_summaries = []   # collect for Telegram notification later

if not video_paths:
    print(f"   No videos found in: {SOURCE_DIR}")
else:
    for vid_path in video_paths:
        vid_name = vid_path.name
        print(f"\n{'='*65}")
        print(f"  Processing video: {vid_name}")
        print(f"{'='*65}")

        cap = cv2.VideoCapture(str(vid_path))
        if not cap.isOpened():
            print(f"\u26a0\ufe0f  Could not open: {vid_name} \u2014 skipping")
            continue

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        effective_fps = fps / FRAME_SKIP
        frames_to_process = total_frames // FRAME_SKIP
        print(f"   {width}x{height} @ {fps:.1f} fps, {total_frames} frames")
        print(f"   Frame skip: {FRAME_SKIP} (processing ~{frames_to_process} frames at ~{effective_fps:.1f} effective FPS)")

        # Output video writer (writes at original fps, only annotated frames)
        out_name = f"{vid_path.stem}_annotated.mp4"
        out_path = os.path.join(OUTPUT_DIR, out_name)
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        writer = cv2.VideoWriter(out_path, fourcc, effective_fps, (width, height))

        # Tracker uses effective FPS (after skipping) for time calculations
        tracker = FeedingTracker(fps=effective_fps)
        last_ts = ""

        for frame_idx in tqdm(range(total_frames), desc=vid_name, unit="frame"):
            ret, frame = cap.read()
            if not ret:
                break

            # Skip frames for faster processing
            if frame_idx % FRAME_SKIP != 0:
                continue

            # OCR timestamp every Nth frame
            if frame_idx % OCR_EVERY_N_FRAMES == 0:
                try:
                    last_ts = extract_timestamp(frame)
                except Exception:
                    pass  # keep last_ts

            # Prepare for inference (IR detection + conversion)
            image_type = detect_image_type(frame)
            inference_img = prepare_for_inference(frame, image_type)

            # YOLO inference
            results = model.predict(
                source=inference_img,
                imgsz=IMGSZ,
                conf=CONF_THRESHOLD,
                iou=IOU_THRESHOLD,
                verbose=False,
            )

            # Parse detections
            detections = parse_results(results)

            # Draw on original frame — boxes only, no labels/percentages
            annotated = draw_boxes(frame, detections, show_label=False)
            writer.write(annotated)

            # Feed to tracker (frame_idx in tracker space = sequential)
            tracker_frame_idx = frame_idx // FRAME_SKIP
            tracker.process_frame(tracker_frame_idx, detections, last_ts, frame)

        cap.release()
        writer.release()
        print(f"\n\u2705 Annotated video saved \u2192 {out_path}")

        # --- Summary ---
        summary = tracker.summarize()
        summary_text = format_feeding_summary(summary, vid_name)
        print(f"\n{summary_text}")

        # --- Save text summary to Google Drive ---
        txt_path = os.path.join(OUTPUT_DIR, f"{vid_path.stem}_summary.txt")
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(summary_text)
        print(f"\u2705 Summary text saved \u2192 {txt_path}")

        # --- Save snapshots ---
        snapshot_paths = []
        for event_name, snap_img in tracker.snapshots.items():
            snap_path = os.path.join(OUTPUT_DIR, f"{vid_path.stem}_{event_name}.jpg")
            cv2.imwrite(snap_path, snap_img)
            snapshot_paths.append(snap_path)
            print(f"\u2705 Snapshot saved \u2192 {snap_path}")

        # --- Timeline chart ---
        plot_video_timeline(tracker, vid_name)

        # Collect for Telegram
        video_summaries.append({
            "video_name": vid_name,
            "summary_text": summary_text,
            "snapshot_paths": snapshot_paths,
            "video_path": out_path,
        })

        # --- Try to display video inline ---
        try:
            display(Video(out_path, embed=True, width=800))
        except Exception:
            print("   (Video display not supported inline; download from Google Drive)")

    print(f"\n\u2705  All video outputs saved to: {OUTPUT_DIR}")

In [ ]:
# ───────────────────────────────────────────────────────────────
# Telegram Notification
# ───────────────────────────────────────────────────────────────
# BOT_TOKEN and MY_CHAT_ID are loaded from Infisical (cell 2 above).


def send_telegram_summary(bot_token, chat_id, summary_text, snapshot_paths, video_path=None):
    """Send feeding summary, snapshots, and annotated video via Telegram Bot API."""
    if not bot_token or not chat_id:
        print("\u26a0\ufe0f  BOT_TOKEN / MY_CHAT_ID not set. Skipping Telegram notification.")
        return False

    base = f"https://api.telegram.org/bot{bot_token}"
    ok = True

    # 1. Send text summary (Telegram limit = 4096 chars)
    r = requests.post(f"{base}/sendMessage", data={
        "chat_id": chat_id,
        "text": summary_text[:4096],
    })
    if r.status_code != 200:
        print(f"\u274c Telegram text error: {r.text[:200]}")
        return False
    print("\u2705 Telegram summary sent")

    # 2. Send each snapshot as a photo
    for path in snapshot_paths:
        try:
            with open(path, "rb") as fh:
                r = requests.post(
                    f"{base}/sendPhoto",
                    data={"chat_id": chat_id},
                    files={"photo": (Path(path).name, fh, "image/jpeg")},
                )
            if r.status_code != 200:
                print(f"\u26a0\ufe0f  Photo failed ({Path(path).name}): {r.text[:100]}")
                ok = False
            else:
                print(f"\u2705 Snapshot sent: {Path(path).name}")
        except Exception as e:
            print(f"\u26a0\ufe0f  Photo error ({Path(path).name}): {e}")
            ok = False

    # 3. Send annotated video (Telegram Bot API limit: 50 MB)
    if video_path and os.path.exists(video_path):
        size_mb = os.path.getsize(video_path) / 1e6
        if size_mb <= 50:
            try:
                with open(video_path, "rb") as fh:
                    r = requests.post(
                        f"{base}/sendVideo",
                        data={"chat_id": chat_id,
                              "caption": f"\U0001f3a5 {Path(video_path).name}"},
                        files={"video": (Path(video_path).name, fh, "video/mp4")},
                        timeout=120,
                    )
                if r.status_code != 200:
                    print(f"\u26a0\ufe0f  Video send failed: {r.text[:100]}")
                    ok = False
                else:
                    print(f"\u2705 Video sent: {Path(video_path).name}")
            except Exception as e:
                print(f"\u26a0\ufe0f  Video error: {e}")
                ok = False
        else:
            requests.post(f"{base}/sendMessage", data={
                "chat_id": chat_id,
                "text": f"\u26a0\ufe0f Video too large for Telegram ({size_mb:.0f} MB). Check Google Drive:\n{video_path}",
            })
            print(f"\u26a0\ufe0f  Video too large ({size_mb:.0f} MB) \u2014 Drive path sent instead")

    return ok


# --- Send all video summaries ---
if video_summaries:
    print(f"\nSending {len(video_summaries)} summary(ies) to Telegram...\n")
    for vs in video_summaries:
        send_telegram_summary(
            BOT_TOKEN,
            MY_CHAT_ID,
            vs["summary_text"],
            vs["snapshot_paths"],
            vs.get("video_path"),
        )
else:
    print("No video summaries to send.")

## Results

All annotated outputs are saved to the output directory on Google Drive.

- **Images**: `{filename}_boxes_only.jpg` + `{filename}_with_conf.jpg`
- **Videos**: `{filename}_annotated.mp4` (boxes only — no labels/percentages)
- **Snapshots**: `{filename}_sanbo_arrival.jpg`, `{filename}_dan_hand_ep0.jpg`, `{filename}_kibble_dispensed_ep0.jpg`, etc.
- **Text summary**: `{filename}_summary.txt` (saved to Google Drive)
- **Charts**: `timeline_{filename}.jpg`, `SUMMARY_TABLE.jpg`

The feeding summary is printed above, saved as `.txt`, and sent to Telegram (if secrets are configured in Infisical).

### Summary format
```
🐱 Fair Feeder Summary — video.mp4
   2024-01-15 18:30:00 → 19:00:00
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📦 Feeding (Dan_hand): 2 attempt(s)
   18:32:15 — ~5 kibble added
   18:45:30 — ~3 kibble added

🐈 Sanbo arrived at 18:35:42
🐈‍⬛ Dan present from 18:30:05

🍽️ Eating estimate:
   Dan:   ~12 kibble (at bowl 5m 30s)
   Sanbo: ~8 kibble  (at bowl 3m 15s)

✋ Dan's hand attempts: 2
```